In [1]:
import sys, importlib.metadata as md
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
for pkg in ["flwr", "fedml", "syft", "opacus", "torch", "torchvision", "numpy", "ray", "flgo"]:
    try:
        print(f"{pkg}: {md.version(pkg)}")
    except Exception:
        print(f"{pkg}: не установлен")

Python: 3.11.9
Executable: C:\Users\nak\PycharmProjects\practika_leto\venv-pysyft\Scripts\python.exe
flwr: не установлен
fedml: не установлен
syft: 0.9.5
opacus: не установлен
torch: не установлен
torchvision: не установлен
numpy: 1.24.4
ray: не установлен
flgo: не установлен


In [1]:
import syft as sy
import numpy as np

In [2]:
server = sy.orchestra.launch(name="hello-datasite", dev_mode=True, reset=True)
admin = server.login(email="info@openmined.org", password="changethis")

CRITICAL:syft.server.server:Hash of the signing key 'ed653...'


Using SQLiteDBConfig and sqlite:///C:\Users\nak\AppData\Local\Temp\syft\6d3dd2af875b4b6b86831fc548339536\db\6d3dd2af875b4b6b86831fc548339536_json.db


SyftInfo: You have launched a development server at http://0.0.0.0:None. It is intended only for local use.

Logged into <hello-datasite: High side Datasite> as <info@openmined.org>


SyftWarning: You are using a default password. Please change the password using `[your_client].account.set_password([new_password])`.

In [3]:
real = np.array([25.0, 30.0, 45.0, 50.0, 20.0])
mock = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
dataset = sy.Dataset(name="ages",
    asset_list=[sy.Asset(name="ages_data", data=real, mock=mock)])
admin.upload_dataset(dataset)

Uploading:   0%|          | 0/1 [00:00<?, ?it/s]

Uploading: ages_data:   0%|          | 0/1 [00:00<?, ?it/s]

Uploading: ages_data: 100%|██████████| 1/1 [00:00<00:00,  5.32it/s]

Uploading: ages_data: 100%|██████████| 1/1 [00:00<00:00,  5.21it/s]

SyftSuccess: Dataset uploaded to 'hello-datasite'. To see the datasets uploaded by a client on this server, use command `[your_client].datasets`

In [4]:
admin.register(name="DS", email="ds@test.com",
               password="pw12345", password_verify="pw12345")
ds = server.login(email="ds@test.com", password="pw12345")

Logged into <hello-datasite: High side Datasite> as <ds@test.com>


In [5]:
asset = ds.datasets["ages"].assets["ages_data"]
print("Исследователь видит mock:", asset.mock)

Исследователь видит mock: [1. 2. 3. 4. 5.]


In [6]:
@sy.syft_function_single_use(data=asset)
def mean_age(data):
    return data.mean()

ds.code.request_code_execution(mean_age)

SyftSuccess: Syft function 'mean_age' successfully created. To add a code request, please create a project using `project = syft.Project(...)`, then use command `project.create_code_request`.

syft.service.request.request.Request

In [7]:
admin.requests[0].approve()

Approving request on change mean_age for datasite hello-datasite


SyftSuccess: Request 82b9dd8355cd4684ad9157c792dc5083 changes applied

In [8]:
res = ds.code.mean_age(data=asset)
print("Результат (среднее скрытых данных):", res.get())

Результат (среднее скрытых данных): 34.0
